In [5]:
import re
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [6]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...


True

### 1. Load the dataset

In [28]:
import pandas as pd

def load_data(file_path: str) -> pd.DataFrame:
    """
    Load SMS spam dataset from a file path.

    Format:
    - Tab-separated file
    - Column 0: label (ham/spam)
    - Column 1: message text
    """
    df = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=["label", "message"]
    )
    return df

In [29]:
df = load_data("SMSSpamCollection")
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [30]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

---

## 2. Preprocessing

### 2.1 Label Preprocessing

In [7]:
def validate_labels(df: pd.DataFrame) -> None:
    """
    Ensure labels contain only 'ham' and 'spam'.
    """
    allowed = {"ham", "spam"}
    unique_labels = set(df["label"].unique())

    if not unique_labels.issubset(allowed):
        raise ValueError(f"Unexpected labels found: {unique_labels}")

In [8]:
def encode_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Encode labels: ham -> 0, spam -> 1
    """
    df = df.copy()
    df["label"] = df["label"].map({"ham": 0, "spam": 1})
    return df

### 2.2 Text Normalization

In [9]:
def to_lowercase(text: str) -> str:
    return text.lower()

In [10]:
def remove_noise(text: str) -> str:
    text = re.sub(r"http\S+|www\S+", "", text)        # URLs
    text = re.sub(r"\S+@\S+", "", text)               # Emails
    text = re.sub(r"\+?\d[\d\s\-]{8,}\d", "", text)   # Phone numbers
    return text

In [11]:
def remove_punctuation(text: str) -> str:
    return re.sub(r"[^a-zA-Z\s]", "", text)

In [12]:
def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

### 2.3 Language Cleanup

In [13]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [14]:
def tokenize(text: str) -> list:
    return word_tokenize(text)

In [15]:
def remove_stopwords(tokens: list) -> list:
    return [word for word in tokens if word not in stop_words]

In [16]:
def lemmatize(tokens: list) -> list:
    return [lemmatizer.lemmatize(word) for word in tokens]

### 2.4 Preprocessing pipeline

In [17]:
def preprocess_text(text: str) -> str:
    """
    Complete preprocessing pipeline for a single SMS.
    """
    text = to_lowercase(text)
    text = remove_noise(text)
    text = remove_punctuation(text)
    text = normalize_whitespace(text)

    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)

    return " ".join(tokens)

In [27]:
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove duplicate SMS messages.
    Keeps the first occurrence.
    """
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates(subset="message")
    after = len(df)

    print(f"Removed {before - after} duplicate messages")

    return df

In [31]:
def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply preprocessing to entire dataframe.
    """
    df = df.copy()

    validate_labels(df)

    df = remove_duplicates(df)
    
    df = encode_labels(df)

    df["clean_message"] = df["message"].apply(preprocess_text)

    return df

In [32]:
df_clean = preprocess_dataframe(df)

df_clean.head()

Removed 403 duplicate messages


,label,message,clean_message
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though


---

## 3. Splitting the data into train, validation and test

In [33]:
from sklearn.model_selection import train_test_split

In [34]:
def split_train_temp(df, test_size=0.3, random_state=42):
    """
    Split data into train and temporary set.
    """
    train_df, temp_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["label"],
        random_state=random_state
    )
    return train_df, temp_df

In [35]:
def split_validation_test(temp_df, random_state=42):
    """
    Split temporary data equally into validation and test sets.
    """
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        stratify=temp_df["label"],
        random_state=random_state
    )
    return val_df, test_df

In [36]:
def save_splits(train_df, val_df, test_df):
    """
    Save train, validation, and test splits to CSV files
    in the current working directory.
    """
    train_df.to_csv("train.csv", index=False)
    val_df.to_csv("validation.csv", index=False)
    test_df.to_csv("test.csv", index=False)

In [37]:
def split_and_save(df):
    """
    Split dataframe into train/validation/test and save to disk.
    """
    train_df, temp_df = split_train_temp(df)
    val_df, test_df = split_validation_test(temp_df)

    save_splits(train_df, val_df, test_df)

    return train_df, val_df, test_df

In [38]:
train_df, val_df, test_df = split_and_save(df_clean)

In [39]:
print(len(train_df), len(val_df), len(test_df))

print(train_df["label"].value_counts(normalize=True))
print(val_df["label"].value_counts(normalize=True))
print(test_df["label"].value_counts(normalize=True))

3618 775 776
label
0    0.873687
1    0.126313
Name: proportion, dtype: float64
label
0    0.873548
1    0.126452
Name: proportion, dtype: float64
label
0    0.873711
1    0.126289
Name: proportion, dtype: float64
